# Analyst kit (N5)

Reads the SQLite history through the `v_*` views, read-only. Needs `pandas` and `matplotlib` in the notebook's
own environment: the app itself does not depend on them.

The same tables as CSV: `python main.py export --out ./export`.

In [ ]:
import sqlite3
import pandas as pd

DB = '../../portfolio_history.db'   # settings.db_path of config.yaml
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
read = lambda view: pd.read_sql(f'SELECT * FROM {view}', con, parse_dates=['day'])

## NAV and flow-adjusted daily return

`v_daily_returns.return` is the Modified Dietz return between two daily closes, the one `GET /api/analytics/quant`
chains into its TWR index. A day with no return (first day, or a capital base too small) carries the index forward.

In [ ]:
returns = read('v_daily_returns').set_index('day')
returns['twr_index'] = 100 * (1 + returns['return'].fillna(0)).cumprod()
ax = returns['nav_usd'].plot(figsize=(9, 3), title='NAV (USD)')
returns[['twr_index']].plot(figsize=(9, 3), title='TWR index, base 100')
returns.tail()

## Positions, external flows, Earn income, trades

In [ ]:
positions = read('v_positions_daily').pivot(index='day', columns='symbol', values='value_usd')
positions.plot.area(figsize=(9, 3), title='Value held per asset (USD)')

flows = read('v_external_flows')
earn = read('v_earn_income_daily').pivot_table(index='day', columns='asset', values='value_usd', aggfunc='sum')
earn.resample('M').sum().plot.bar(stacked=True, figsize=(9, 3), title='Earn income per month (USD)')

trades = read('v_trades')
trades.tail()